In [1]:
# Author: Ryan Benac CEMVR EC-D
# Last Update: 1/20/2026
# This script uses python playwright to upload data to EMS REST API from a spreadsheet
##################################################################################################################################
print(f"Importing modules...")
import pandas as pd # used to manage datatable
from playwright.sync_api import sync_playwright, TimeoutError # used to interact with EMS
import requests, json
from urllib.parse import quote
import datetime
import urllib3

# Disable the InsecureRequestWarning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


# variables
#baseEMSURL = "https://ems-test.cwbi.us"
# The 'r' before the string treats backslashes as literal characters
#file_path = r"C:\Workspace\LOCAL SANDBOX\EMS Rest API\Bulk Insert Tasks to EMS\TEST_DATASETS.xlsx" 
#sheet_name = "compiled"


# variables
baseEMSURL = "https://ems.sec.usace.army.mil/"
# The 'r' before the string treats backslashes as literal characters
file_path = r"C:\Workspace\LOCAL SANDBOX\EMS Rest API\Bulk Insert Tasks to EMS\NEW Compiled Worksheet of CR-FRM EMS Products w P6 Tasks Categorized.xlsx"
sheet_name = "compiled"

print('Modules imported...')

Importing modules...
Modules imported...


In [ ]:
##################################################################################################################################
# STEP 1: import spreadsheet as datatable
print(f"Importing spreadsheet at {file_path}...")
try:
    # Read the specified sheet from the Excel file into a pandas DataFrame
    # The 'engine='openpyxl'' is required for .xlsx files
    # dtype=str forces all columns to be read as text initially
    df = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl', dtype=str)

    # Print a success message
    print(f"Successfully imported data from sheet '{sheet_name}'.")

except FileNotFoundError:
    print(f"Error: The file was not found at the path: {file_path}")
    raise
except Exception as e:
    # Catch other potential errors, such as the sheet not existing
    print(f"An error occurred: {e}")
    raise

# Validate required columns
required_columns = ['product_id', 'product_title', 'master_task', 'wbs_sub_id', 'wbs_id', 'task_name', 'start_date', 'end_date']
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print(f"Error: The following required columns are missing: {', '.join(missing_columns)}")
    print(f"Available columns: {', '.join(df.columns.tolist())}")
    raise ValueError(f"Missing required columns: {', '.join(missing_columns)}")
else:
    print(f"✓ All required columns are present: {', '.join(required_columns)}")

# Clean up text columns - remove trailing .0 from numeric-looking strings BUT preserve other trailing zeros
text_columns = ['product_id', 'master_task', 'wbs_sub_id', 'wbs_id']
for col in text_columns:
    if col in df.columns:
        # Only remove .0 specifically (e.g., '2.0' becomes '2', but '2.3.10' stays '2.3.10')
        def clean_trailing_dot_zero(x):
            if pd.notna(x):
                s = str(x)
                # Only remove .0 at the end, not other trailing zeros
                if s.endswith('.0'):
                    return s[:-2]
                return s
            return x
        
        df[col] = df[col].apply(clean_trailing_dot_zero)
        print(f"✓ Cleaned {col} column (removed trailing .0 only)")

# Format dates as yyyy-mm-dd
# Convert date columns to datetime objects, coercing errors to NaT (Not a Time)
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')

# Format the datetime objects into the desired 'yyyy-mm-dd' string format
df['start_date'] = df['start_date'].dt.strftime('%Y-%m-%d')
df['end_date'] = df['end_date'].dt.strftime('%Y-%m-%d')

print(f"✓ Formatted date columns as yyyy-mm-dd")

# Convert forward slashes to backward slashes in task_name
df['task_name'] = df['task_name'].astype(str).str.replace('/', '\\', regex=False)
print(f"✓ Converted forward slashes to backslashes in task_name")

# URL-encode task_name values up-front and store in a column (encode slashes too)
df['task_name_encoded'] = df['task_name'].apply(lambda x: quote(str(x), safe='') if pd.notna(x) else '')
print(f"✓ Created task_name_encoded column")

# Display the first 5 rows of the imported data to verify
#print("\nFirst 5 rows of the DataTable:")
#print(df.head())
print(f"\nDataFrame shape: {df.shape[0]} rows, {df.shape[1]} columns")

Importing spreadsheet at C:\Workspace\LOCAL SANDBOX\EMS Rest API\Bulk Insert Tasks to EMS\NEW Compiled Worksheet of CR-FRM EMS Products w P6 Tasks Categorized.xlsx...
Successfully imported data from sheet 'compiled'.
✓ All required columns are present: product_id, product_title, master_task, wbs_sub_id, wbs_id, task_name, start_date, end_date
✓ Cleaned product_id column (removed trailing .0)
✓ Cleaned master_task column (removed trailing .0)
✓ Cleaned wbs_sub_id column (removed trailing .0)
✓ Cleaned wbs_id column (removed trailing .0)
✓ Formatted date columns as yyyy-mm-dd
✓ Converted forward slashes to backslashes in task_name
✓ Created task_name_encoded column

DataFrame shape: 860 rows, 15 columns


In [ ]:
sessionID = "9230553"
product_id = "427476"  
time_out = 30 # seconds
success_code = 200

unique_product_ids = df['product_id'].unique()
print(f"\nFound {len(unique_product_ids)} unique products to process.")

# filter to only current product id (both are now strings)
sub_dataset = df[df['product_id'] == product_id]

# this section just gets the master task id to add new tasks to
print(f"\nProcessing Product ID {product_id} ({len(sub_dataset)} rows).")

def fetch_scope_data(session_id, prod_id):
    """Fetch scope data from API"""
    scope_url = f"{baseEMSURL}/api/SCOPE/SSB_SCOPE/{session_id}/{prod_id}"
    try:
        print(f"   - Requesting scope: {scope_url}")
        resp = requests.get(scope_url, timeout=time_out, verify=False)
        if resp.status_code == 200:
            data = resp.json()
            print(f"   - Success from {scope_url} (items={len(data)})")
            return data
        else:
            print(f"   - Scope endpoint returned status {resp.status_code}: {scope_url}")
            return None
    except Exception as e:
        print(f"   - Error requesting {scope_url}: {e}")
        return None

def calculate_subtask_dates(wbs_prefix, scope_data):
    """
    Calculate min start date and max end date for all subtasks under a given WBS prefix.
    For example, if wbs_prefix is '2.1', this will look at all tasks starting with '2.1.'
    and return the earliest start and latest end date.
    """
    subtasks = [s for s in scope_data if str(s.get('wbs', '')).startswith(f"{wbs_prefix}.")]
    
    if not subtasks:
        return None, None
    
    start_dates = []
    end_dates = []
    
    for task in subtasks:
        # Try to get dates from the task data (adjust field names as needed based on API)
        task_start = task.get('startDate') or task.get('start_date') or task.get('plannedStartDate')
        task_end = task.get('endDate') or task.get('end_date') or task.get('plannedEndDate')
        
        if task_start:
            try:
                start_dates.append(pd.to_datetime(task_start))
            except:
                pass
        
        if task_end:
            try:
                end_dates.append(pd.to_datetime(task_end))
            except:
                pass
    
    min_start = min(start_dates).strftime('%Y-%m-%d') if start_dates else None
    max_end = max(end_dates).strftime('%Y-%m-%d') if end_dates else None
    
    return min_start, max_end

def has_subtasks(wbs_id, dataset):
    """Check if a WBS ID has any subtasks in the dataset"""
    wbs_prefix = f"{wbs_id}."
    subtasks = dataset[dataset['wbs_id'].astype(str).str.startswith(wbs_prefix)]
    return len(subtasks) > 0

scope_url = f"{baseEMSURL}/api/SCOPE/SSB_SCOPE/{sessionID}/{product_id}"
scope_data = fetch_scope_data(sessionID, product_id)

if scope_data is None:
    print(f"   - Failed to retrieve scope for product {product_id}; skipping this product.")
    # continue
else:
    # there should only be 1 master task per product
    master_wbs = str(sub_dataset['master_task'].unique()[0])
    print(f"   - Looking for master_wbs: '{master_wbs}'")

    wbs1_entries = [s for s in scope_data if str(s.get('wbs')) == master_wbs]
    print(f"   - Found {len(wbs1_entries)} matching entries")

    if not wbs1_entries:
        print(f"   - No scope entries with wbs == '{master_wbs}' found. Skipping this product.")
        # Debug: print available WBS values
        available_wbs = [s.get('wbs') for s in scope_data if s.get('wbs')]
        print(f"   - Available WBS values in scope_data: {available_wbs}")
        # continue
    else:
        master_task_id = wbs1_entries[0].get('id')
        taskName = wbs1_entries[0].get('lineItem')
        
        print(f"   - Master task: id={master_task_id} and lineItem={taskName}")


Found 15 unique products to process.

Processing Product ID 427476 (79 rows).
   - Requesting scope: https://ems.sec.usace.army.mil//api/SCOPE/SSB_SCOPE/9230553/427476
   - Success from https://ems.sec.usace.army.mil//api/SCOPE/SSB_SCOPE/9230553/427476 (items=17)
   - Looking for master_wbs: '2'
   - Found 1 matching entries
   - Master task: id=7112588 and lineItem=CR-FRM Program P6 Task Structure


In [99]:
print(master_wbs)

2


In [ ]:
import time
import sys

# iterates over each row in the filtered dataset and creates task then adds start/end dates
insert_success_count = 0
insert_error = ""
start_success_count = 0
start_error = ""
end_success_count = 0
end_error = ""
skipped_count = 0
skipped_rows = ""
skipped_date_updates = 0
this_index = -1

# Retry settings
MAX_RETRIES = 5
RETRY_DELAY = 5  # seconds between retries
EXTENDED_RETRY_DELAY = 10  # doubled delay for last 2 attempts
POST_INSERT_DELAY = 3  # seconds to wait after successful insert

# Create a dictionary to cache WBS to ID mappings for quick parent lookups
wbs_to_id_map = {}
# Create a set to track existing WBS IDs for quick lookup
existing_wbs_set = set()
# Create a dictionary to track existing task names by WBS
existing_tasks_map = {}
# Create a dictionary to track existing dates by task ID
existing_dates_map = {}

def rebuild_maps_from_scope(scope_data):
    """Rebuild all mapping dictionaries from scope data"""
    wbs_map = {}
    wbs_set = set()
    tasks_map = {}
    
    for item in scope_data:
        wbs_value = str(item.get('wbs'))
        task_name_value = item.get('lineItem', '')
        task_id_value = item.get('id')
        wbs_map[wbs_value] = task_id_value
        wbs_set.add(wbs_value)
        tasks_map[wbs_value] = task_name_value
    
    return wbs_map, wbs_set, tasks_map

if scope_data:
    wbs_to_id_map, existing_wbs_set, existing_tasks_map = rebuild_maps_from_scope(scope_data)
    print(f"   - Built WBS-to-ID map with {len(wbs_to_id_map)} entries")
    print(f"   - Existing WBS IDs: {sorted(existing_wbs_set)}")

def get_existing_dates(session_id, prod_id, t_id):
    """Fetch existing start and end dates for a task"""
    try:
        # This assumes there's an API to get task details - adjust URL as needed
        task_url = f"{baseEMSURL}/api/ssb_task_overrides/{session_id}/{prod_id}/{t_id}"
        resp = requests.get(task_url, timeout=time_out, verify=False)
        if resp.status_code == 200:
            task_data = resp.json()
            # Adjust these field names based on actual API response
            existing_start = task_data.get('startDate', None)
            existing_end = task_data.get('endDate', None)
            return existing_start, existing_end
    except:
        pass
    return None, None

for index, row in sub_dataset.iterrows():
    this_index = this_index + 1
    print(f"\n{datetime.datetime.now()} Processing row {this_index + 1}/{len(sub_dataset)}...")
    
    task_name = row['task_name_encoded']
    task_name_display = row['task_name']
    wbs_id = str(row['wbs_id'])  # Full WBS like "2.1.1" or "2.3.10"
    
    # CHECK IF TASK ALREADY EXISTS (both WBS and name match)
    if wbs_id in existing_wbs_set:
        existing_task = [s for s in scope_data if str(s.get('wbs')) == wbs_id]
        if existing_task:
            existing_task_name = existing_task[0].get('lineItem', '')
            existing_task_id = existing_task[0].get('id')
            
            # Check if both WBS and name match
            if existing_task_name == task_name_display:
                print(f"     - Task already exists: WBS={wbs_id}, ID={existing_task_id}, Name='{existing_task_name}'. Skipping insert...")
                skipped_count += 1
                skipped_rows = f"{skipped_rows}, {this_index+1}" if skipped_rows else str(this_index+1)
                
                # Check if dates need updating
                task_id = existing_task_id
                update_dates = True
            else:
                print(f"     - WARNING: WBS {wbs_id} exists but with different name:")
                print(f"       Existing: '{existing_task_name}'")
                print(f"       New:      '{task_name_display}'")
                print(f"     - Will attempt to insert as new task.")
                update_dates = False
        else:
            update_dates = False
    else:
        update_dates = False
        task_id = None
        
        # Determine parent WBS and parent task ID
        wbs_parts = wbs_id.split('.')
        
        if len(wbs_parts) == 1:
            parent_wbs = master_wbs
            parent_task_id = master_task_id
            subtask_number = wbs_parts[0]
        else:
            parent_wbs = '.'.join(wbs_parts[:-1])
            subtask_number = wbs_parts[-1]  # This now correctly preserves '10', '20', etc.
            parent_task_id = wbs_to_id_map.get(parent_wbs)
            
            if parent_task_id is None:
                print(f"     - ✗ CRITICAL ERROR: Parent WBS '{parent_wbs}' not found in scope data. Cannot insert task.")
                print(f"     - STOPPING SCRIPT. Please resolve this issue before continuing.")
                insert_error = f"{insert_error}, {this_index+1}" if insert_error else str(this_index+1)
                sys.exit(1)

        print(f"     - Row {this_index + 1}: task='{task_name_display}', wbs_id={wbs_id}, parent_wbs={parent_wbs}, parent_task_id={parent_task_id}, subtask_number={subtask_number}")

        if parent_task_id is None:
            print("     - ✗ CRITICAL ERROR: No parent_task_id available.")
            print("     - STOPPING SCRIPT.")
            insert_error = f"{insert_error}, {this_index+1}" if insert_error else str(this_index+1)
            sys.exit(1)
        elif not task_name:
            print("     - ✗ CRITICAL ERROR: task_name is empty.")
            print("     - STOPPING SCRIPT.")
            insert_error = f"{insert_error}, {this_index+1}" if insert_error else str(this_index+1)
            sys.exit(1)
        
        # RETRY LOOP FOR INSERT
        insert_url = f"{baseEMSURL}/api/ssb_wbs/insert/{sessionID}/{product_id}/{parent_task_id}/{task_name}/{subtask_number}"
        insert_successful = False
        
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                # Determine delay based on attempt number
                if attempt > MAX_RETRIES - 2:  # Last 2 attempts
                    current_delay = EXTENDED_RETRY_DELAY
                    delay_label = f"{current_delay}s (extended)"
                else:
                    current_delay = RETRY_DELAY
                    delay_label = f"{current_delay}s"
                
                print(f"     - INSERT ATTEMPT {attempt}/{MAX_RETRIES}: {insert_url}")
                resp = requests.get(insert_url, timeout=time_out, verify=False)
                print(f"     - Insert request -> status {resp.status_code}")
                
                if resp.status_code == success_code:
                    new_scope_data = resp.json()
                    
                    # Update scope_data and maps
                    scope_data = new_scope_data
                    wbs_to_id_map, existing_wbs_set, existing_tasks_map = rebuild_maps_from_scope(scope_data)
                    
                    # VERIFY the task was created with correct WBS
                    created_task_entries = [s for s in scope_data if str(s.get('wbs')) == wbs_id]
                    
                    if created_task_entries:
                        task_id = created_task_entries[0].get('id')
                        actual_wbs = created_task_entries[0].get('wbs')
                        actual_name = created_task_entries[0].get('lineItem', '')
                        print(f"     - ✓ VERIFIED: Created task ID: {task_id} with WBS {actual_wbs}")
                        
                        if str(actual_wbs) == wbs_id:
                            insert_success_count += 1
                            insert_successful = True
                            update_dates = True
                            print(f"     - Pausing {POST_INSERT_DELAY} seconds after successful insert...")
                            time.sleep(POST_INSERT_DELAY)
                            break
                        else:
                            print(f"     - ✗ WARNING: WBS mismatch! Expected {wbs_id}, got {actual_wbs}")
                            print(f"     - This suggests the system assigned a different ID.")
                            print(f"     - Re-fetching scope to verify if task was actually created...")
                            
                            # Re-fetch scope to see if the task exists with the expected WBS
                            verification_scope = fetch_scope_data(sessionID, product_id)
                            if verification_scope:
                                scope_data = verification_scope
                                wbs_to_id_map, existing_wbs_set, existing_tasks_map = rebuild_maps_from_scope(scope_data)
                                
                                # Check again for the expected WBS
                                reverify_task = [s for s in scope_data if str(s.get('wbs')) == wbs_id]
                                if reverify_task:
                                    task_id = reverify_task[0].get('id')
                                    actual_wbs = reverify_task[0].get('wbs')
                                    print(f"     - ✓ REVERIFIED: Task exists with WBS {actual_wbs}, ID {task_id}")
                                    insert_success_count += 1
                                    insert_successful = True
                                    update_dates = True
                                    print(f"     - Pausing {POST_INSERT_DELAY} seconds after successful verification...")
                                    time.sleep(POST_INSERT_DELAY)
                                    break
                                else:
                                    print(f"     - Task with WBS {wbs_id} still not found after re-fetch.")
                                    if attempt < MAX_RETRIES:
                                        print(f"     - Retrying in {delay_label}...")
                                        time.sleep(current_delay)
                                    else:
                                        print(f"     - ✗ CRITICAL ERROR: Failed after {MAX_RETRIES} attempts.")
                                        print(f"     - STOPPING SCRIPT to prevent further data corruption.")
                                        insert_error = f"{insert_error}, {this_index+1}" if insert_error else str(this_index+1)
                                        sys.exit(1)
                            else:
                                if attempt < MAX_RETRIES:
                                    print(f"     - Retrying in {delay_label}...")
                                    time.sleep(current_delay)
                                else:
                                    print(f"     - ✗ CRITICAL ERROR: Failed after {MAX_RETRIES} attempts.")
                                    print(f"     - STOPPING SCRIPT.")
                                    insert_error = f"{insert_error}, {this_index+1}" if insert_error else str(this_index+1)
                                    sys.exit(1)
                    else:
                        print(f"     - ✗ ERROR: Task not found in response with WBS {wbs_id}")
                        print(f"     - Available WBS in response: {sorted([str(s.get('wbs')) for s in scope_data if s.get('wbs')])}")
                        
                        # Re-fetch scope to verify
                        print(f"     - Re-fetching scope to verify if task was actually created...")
                        verification_scope = fetch_scope_data(sessionID, product_id)
                        if verification_scope:
                            scope_data = verification_scope
                            wbs_to_id_map, existing_wbs_set, existing_tasks_map = rebuild_maps_from_scope(scope_data)
                            
                            # Check for the expected WBS
                            reverify_task = [s for s in scope_data if str(s.get('wbs')) == wbs_id]
                            if reverify_task:
                                task_id = reverify_task[0].get('id')
                                actual_wbs = reverify_task[0].get('wbs')
                                print(f"     - ✓ REVERIFIED: Task exists with WBS {actual_wbs}, ID {task_id}")
                                insert_success_count += 1
                                insert_successful = True
                                update_dates = True
                                print(f"     - Pausing {POST_INSERT_DELAY} seconds after successful verification...")
                                time.sleep(POST_INSERT_DELAY)
                                break
                            else:
                                print(f"     - Task with WBS {wbs_id} still not found after re-fetch.")
                                if attempt < MAX_RETRIES:
                                    print(f"     - Retrying in {delay_label}...")
                                    time.sleep(current_delay)
                                else:
                                    print(f"     - ✗ CRITICAL ERROR: Failed after {MAX_RETRIES} attempts.")
                                    print(f"     - STOPPING SCRIPT.")
                                    insert_error = f"{insert_error}, {this_index+1}" if insert_error else str(this_index+1)
                                    sys.exit(1)
                        else:
                            if attempt < MAX_RETRIES:
                                print(f"     - Retrying in {delay_label}...")
                                time.sleep(current_delay)
                            else:
                                print(f"     - ✗ CRITICAL ERROR: Failed after {MAX_RETRIES} attempts.")
                                print(f"     - STOPPING SCRIPT.")
                                insert_error = f"{insert_error}, {this_index+1}" if insert_error else str(this_index+1)
                                sys.exit(1)
                else:
                    print(f"     - Insert returned status {resp.status_code}")
                    if attempt < MAX_RETRIES:
                        print(f"     - Retrying in {delay_label}...")
                        time.sleep(current_delay)
                    else:
                        print(f"     - ✗ CRITICAL ERROR: Failed after {MAX_RETRIES} attempts with status {resp.status_code}.")
                        print(f"     - STOPPING SCRIPT.")
                        insert_error = f"{insert_error}, {this_index+1}" if insert_error else str(this_index+1)
                        sys.exit(1)
                        
            except Exception as e:
                print(f"     - Insert request failed: {e}")
                if attempt < MAX_RETRIES:
                    print(f"     - Retrying in {delay_label}...")
                    time.sleep(current_delay)
                else:
                    print(f"     - ✗ CRITICAL ERROR: Failed after {MAX_RETRIES} attempts with exception: {e}")
                    print(f"     - STOPPING SCRIPT.")
                    insert_error = f"{insert_error}, {this_index+1}" if insert_error else str(this_index+1)
                    sys.exit(1)
        
        if not insert_successful:
            print(f"     - ✗ CRITICAL ERROR: Failed to insert after {MAX_RETRIES} attempts.")
            print(f"     - STOPPING SCRIPT.")
            insert_error = f"{insert_error}, {this_index+1}" if insert_error else str(this_index+1)
            sys.exit(1)
    
    # STEP 2: CHECK AND INSERT START AND END DATES (only if we have a valid task_id)
    if update_dates and task_id:
        try:
            start_date = row['start_date']
            end_date = row['end_date']
            
            # Check if this task has subtasks in the dataset
            has_children = has_subtasks(wbs_id, sub_dataset)
            
            # If no dates in spreadsheet and task has subtasks, calculate from children
            if (pd.isna(start_date) or pd.isna(end_date) or start_date == 'NaT' or end_date == 'NaT') and has_children:
                print(f"     - No dates in spreadsheet, but task has subtasks. Calculating from children...")
                calc_start, calc_end = calculate_subtask_dates(wbs_id, scope_data)
                
                if calc_start and calc_end:
                    print(f"     - Calculated dates from subtasks: Start={calc_start}, End={calc_end}")
                    start_date = calc_start
                    end_date = calc_end
                else:
                    print(f"     - Could not calculate dates from subtasks (subtasks may not have dates yet).")
        
            if pd.isna(start_date) or pd.isna(end_date):
                print("     - No start or end date available; skipping date insert for this row.")
                start_error = f"{start_error}, {this_index+1}" if start_error else str(this_index+1)
                end_error = f"{end_error}, {this_index+1}" if end_error else str(this_index+1)
            elif not start_date or not end_date or start_date == 'NaT' or end_date == 'NaT':
                print("     - Start or end date is empty/invalid; skipping date insert for this row.")
                start_error = f"{start_error}, {this_index+1}" if start_error else str(this_index+1)
                end_error = f"{end_error}, {this_index+1}" if end_error else str(this_index+1)
            else:
                # Get existing dates to check if update is needed
                existing_start, existing_end = get_existing_dates(sessionID, product_id, task_id)
                
                # Normalize dates for comparison (remove time component if present)
                if existing_start:
                    existing_start_normalized = str(existing_start).split('T')[0] if 'T' in str(existing_start) else str(existing_start).split(' ')[0]
                else:
                    existing_start_normalized = None
                    
                if existing_end:
                    existing_end_normalized = str(existing_end).split('T')[0] if 'T' in str(existing_end) else str(existing_end).split(' ')[0]
                else:
                    existing_end_normalized = None
                
                start_date_str = str(start_date)
                end_date_str = str(end_date)
                
                # Check if start date needs updating
                need_start_update = (existing_start_normalized != start_date_str)
                # Check if end date needs updating
                need_end_update = (existing_end_normalized != end_date_str)
                
                if not need_start_update and not need_end_update:
                    print(f"     - Dates already match (Start: {start_date_str}, End: {end_date_str}). Skipping date updates.")
                    skipped_date_updates += 1
                    start_success_count += 1  # Count as success since dates are correct
                    end_success_count += 1
                else:
                    if need_start_update:
                        print(f"     - Start date needs update: {existing_start_normalized} -> {start_date_str}")
                    else:
                        print(f"     - Start date already correct: {start_date_str}")
                        
                    if need_end_update:
                        print(f"     - End date needs update: {existing_end_normalized} -> {end_date_str}")
                    else:
                        print(f"     - End date already correct: {end_date_str}")
                    
                    # RETRY LOOP FOR START DATE (only if needed)
                    if need_start_update:
                        insert_start_url = f"{baseEMSURL}/api/ssb_task_overrides/updateSTARTDATE/{sessionID}/{product_id}/{task_id}/{start_date}"
                        start_successful = False
                        
                        for attempt in range(1, MAX_RETRIES + 1):
                            try:
                                # Determine delay based on attempt number
                                if attempt > MAX_RETRIES - 2:  # Last 2 attempts
                                    current_delay = EXTENDED_RETRY_DELAY
                                    delay_label = f"{current_delay}s (extended)"
                                else:
                                    current_delay = RETRY_DELAY
                                    delay_label = f"{current_delay}s"
                                
                                print(f"     - START DATE ATTEMPT {attempt}/{MAX_RETRIES}")
                                resp_start = requests.get(insert_start_url, timeout=time_out, verify=False)
                                print(f"     - Insert start date request -> status {resp_start.status_code}")
                                
                                if resp_start.status_code == success_code:
                                    start_success_count += 1
                                    start_successful = True
                                    break
                                else:
                                    if attempt < MAX_RETRIES:
                                        print(f"     - Retrying start date in {delay_label}...")
                                        time.sleep(current_delay)
                                    else:
                                        print(f"     - ✗ CRITICAL ERROR: Failed to insert start date after {MAX_RETRIES} attempts.")
                                        print(f"     - STOPPING SCRIPT.")
                                        start_error = f"{start_error}, {this_index+1}" if start_error else str(this_index+1)
                                        sys.exit(1)
                            except Exception as e:
                                print(f"     - Insert start date request failed: {e}")
                                if attempt < MAX_RETRIES:
                                    print(f"     - Retrying start date in {delay_label}...")
                                    time.sleep(current_delay)
                                else:
                                    print(f"     - ✗ CRITICAL ERROR: Failed to insert start date after {MAX_RETRIES} attempts: {e}")
                                    print(f"     - STOPPING SCRIPT.")
                                    start_error = f"{start_error}, {this_index+1}" if start_error else str(this_index+1)
                                    sys.exit(1)
                        
                        if not start_successful:
                            print(f"     - ✗ CRITICAL ERROR: Start date insert failed.")
                            print(f"     - STOPPING SCRIPT.")
                            start_error = f"{start_error}, {this_index+1}" if start_error else str(this_index+1)
                            sys.exit(1)
                    else:
                        start_success_count += 1  # Count as success since date is already correct

                    # RETRY LOOP FOR END DATE (only if needed)
                    if need_end_update:
                        insert_end_url = f"{baseEMSURL}/api/ssb_task_overrides/updateENDDATE/{sessionID}/{product_id}/{task_id}/{end_date}"
                        end_successful = False
                        
                        for attempt in range(1, MAX_RETRIES + 1):
                            try:
                                # Determine delay based on attempt number
                                if attempt > MAX_RETRIES - 2:  # Last 2 attempts
                                    current_delay = EXTENDED_RETRY_DELAY
                                    delay_label = f"{current_delay}s (extended)"
                                else:
                                    current_delay = RETRY_DELAY
                                    delay_label = f"{current_delay}s"
                                
                                print(f"     - END DATE ATTEMPT {attempt}/{MAX_RETRIES}")
                                resp_end = requests.get(insert_end_url, timeout=time_out, verify=False)
                                print(f"     - Insert end date request -> status {resp_end.status_code}")
                                
                                if resp_end.status_code == success_code:
                                    end_success_count += 1
                                    end_successful = True
                                    break
                                else:
                                    if attempt < MAX_RETRIES:
                                        print(f"     - Retrying end date in {delay_label}...")
                                        time.sleep(current_delay)
                                    else:
                                        print(f"     - ✗ CRITICAL ERROR: Failed to insert end date after {MAX_RETRIES} attempts.")
                                        print(f"     - STOPPING SCRIPT.")
                                        end_error = f"{end_error}, {this_index+1}" if end_error else str(this_index+1)
                                        sys.exit(1)
                            except Exception as e:
                                print(f"     - Insert end date request failed: {e}")
                                if attempt < MAX_RETRIES:
                                    print(f"     - Retrying end date in {delay_label}...")
                                    time.sleep(current_delay)
                                else:
                                    print(f"     - ✗ CRITICAL ERROR: Failed to insert end date after {MAX_RETRIES} attempts: {e}")
                                    print(f"     - STOPPING SCRIPT.")
                                    end_error = f"{end_error}, {this_index+1}" if end_error else str(this_index+1)
                                    sys.exit(1)
                        
                        if not end_successful:
                            print(f"     - ✗ CRITICAL ERROR: End date insert failed.")
                            print(f"     - STOPPING SCRIPT.")
                            end_error = f"{end_error}, {this_index+1}" if end_error else str(this_index+1)
                            sys.exit(1)
                    else:
                        end_success_count += 1  # Count as success since date is already correct
        
        except Exception as e:
            print(f"     - ✗ CRITICAL ERROR processing dates for row {this_index + 1}: {e}")
            print(f"     - STOPPING SCRIPT.")
            start_error = f"{start_error}, {this_index+1}" if start_error else str(this_index+1)
            end_error = f"{end_error}, {this_index+1}" if end_error else str(this_index+1)
            sys.exit(1)

# FINAL SUMMARY
print(f"\n\n===== FINAL SUMMARY =====")
print(f"✓ ALL ROWS PROCESSED SUCCESSFULLY!")
print(f"Total rows processed for product {product_id}: {len(sub_dataset)}")
print(f" - Tasks skipped (already exist): {skipped_count}")
print(f" - Skipped rows: {skipped_rows if skipped_rows else 'None'}")
print(f" - Date updates skipped (already correct): {skipped_date_updates}")
print(f" - Inserted tasks successfully: {insert_success_count}")
print(f" - Inserted/verified start dates successfully: {start_success_count}")
print(f" - Inserted/verified end dates successfully: {end_success_count}")

   - Built WBS-to-ID map with 17 entries
   - Existing WBS IDs: ['1', '1.1', '1.10', '1.11', '1.12', '1.13', '1.14', '1.15', '1.2', '1.3', '1.4', '1.5', '1.6', '1.7', '1.8', '1.9', '2']

2026-03-17 08:10:45.146577 Processing row 1/79...
     - Row 1: task='Real Estate', wbs_id=2.1, parent_wbs=2, parent_task_id=7112588, subtask_number=1
     - INSERT ATTEMPT 1/5: https://ems.sec.usace.army.mil//api/ssb_wbs/insert/9230553/427476/7112588/Real%20Estate/1
     - Insert request -> status 200
     - ✓ VERIFIED: Created task ID: 7116511 with WBS 2.1
     - Pausing 3 seconds after successful insert...
     - No start or end date available; skipping date insert for this row.

2026-03-17 08:10:50.931735 Processing row 2/79...
     - Row 2: task='LTC2R-RE1000__Existing Easement Map(s) - Draw each easement & Map(s)', wbs_id=2.1.1, parent_wbs=2.1, parent_task_id=7116511, subtask_number=1
     - INSERT ATTEMPT 1/5: https://ems.sec.usace.army.mil//api/ssb_wbs/insert/9230553/427476/7116511/LTC2R-RE1000